# ThinBook Pension: run a slice of the sweep on Colab (backup machine)

1. Set the settings in the next cell (repo URL, which algorithm and configs, workers).
2. Run every cell in order. Results are copied to Google Drive every 5 minutes and at the end.
3. To merge with the laptop: download the Drive folder and copy its `logs/sweep/` and `models/`
   folders into the project root. The layout is identical, and the sweep runner skips any run
   whose summary already exists, so nothing is run twice.

Colab only needs the code: the game's settings (`configs/`) are in the repo; no licensed data is used for training.

In [ ]:
import os

WORKERS = os.cpu_count()

print("CPU cores on this Colab machine:", WORKERS)

In [ ]:
REPO_URL = "https://github.com/michael-alu/michael_nwuju_rl_summative.git"  # private repo: use https://<token>@github.com/...

BRANCH = "main"

ALGO = "dqn"          # ppo, a2c, reinforce, dqn or all

CONFIGS = "all"       # all, one index such as 3, or a range such as 0-4

FRAMES = 1_000_000    # keep 1,000,000 so Colab runs match the laptop runs

DRIVE_FOLDER = "/content/drive/MyDrive/thinbook_sweep"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

os.makedirs(DRIVE_FOLDER, exist_ok=True)

In [ ]:
%cd /content

!test -d michael_nwuju_rl_summative || git clone --branch {BRANCH} {REPO_URL} michael_nwuju_rl_summative

%cd /content/michael_nwuju_rl_summative

!git pull

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]

!uv --version

!uv sync

Restore earlier results from Drive, so a restarted session resumes instead of starting over.

In [ ]:
!mkdir -p logs/sweep models

!rsync -a {DRIVE_FOLDER}/logs/sweep/ logs/sweep/ 2>/dev/null || true

!rsync -a {DRIVE_FOLDER}/models/ models/ 2>/dev/null || true

!uv run python -m training.sweep --algo {ALGO} --configs {CONFIGS} --status --workers {WORKERS}

Optional: time one run before committing to a slice. This trains one config for 48,000 frames and prints
seconds per 100,000 frames. Multiply by 10 for one full 1,000,000-frame run. On the laptop the same test gave
about 140 seconds per 100,000 frames for one run alone, and about 280 with 6 runs in parallel.

In [ ]:
import subprocess, time

start = time.time()

subprocess.run([
    "uv", "run", "python", "-m", "training.pg_training", "--name", "colab_timing", "--frames", "48000",
    "--log-dir", "/tmp/colab_timing", "--model-dir", "/tmp/colab_timing"], check=True, stdout=subprocess.DEVNULL
)

seconds = time.time() - start

print(f"{seconds:.0f} s for 48,000 frames -> {seconds * 100_000 / 48_000:.0f} s per 100,000 frames, about {seconds * 1_000_000 / 48_000 / 60:.0f} min per full run (one at a time)")

Run the slice. Results are copied to Drive every 5 minutes while it runs.

In [ ]:
import subprocess, time

command = [
    "uv", "run", "python", "-m", "training.sweep", "--algo", ALGO, "--configs", CONFIGS,
    "--workers", str(WORKERS), "--frames", str(FRAMES)
]

sweep = subprocess.Popen(command, stdout=open("sweep_colab.log", "w"), stderr=subprocess.STDOUT)

def copy_to_drive():
    os.system(f"rsync -a logs/sweep/ {DRIVE_FOLDER}/logs/sweep/ && rsync -a models/ {DRIVE_FOLDER}/models/ && cp sweep_colab.log {DRIVE_FOLDER}/")

while sweep.poll() is None:
    time.sleep(300)
    
    copy_to_drive()
    
    print(time.strftime("%H:%M"), open("sweep_colab.log").read().strip().splitlines()[-1])

copy_to_drive()

print("Finished with exit code", sweep.returncode)

In [ ]:
!uv run python -m training.sweep --algo {ALGO} --configs {CONFIGS} --status --workers {WORKERS}

!ls -R {DRIVE_FOLDER} | head -50